# Cost-Quality Pareto: Finding the Optimal Model

Based on: [Cost and Accuracy in Multi-Agent Systems](https://arxiv.org/abs/2601.07978) (Jan 2026)

## The Concept: Pareto Dominance

When comparing models on two dimensions (quality and cost), some models are strictly better than others. **Pareto dominance** is a precise way to identify which models are worth considering:

**Model A dominates Model B** if Model A is both cheaper AND higher quality (or equal in one and better in the other). A dominated model is never the right choice — there is always a better option.

The **Pareto frontier** is the set of models that are NOT dominated by any other model. These are the only models worth considering, because each one represents the best quality available at its price point.

```
Quality ↑
  1.0  │             * GPT-4o (Pareto: highest quality)
       │
  0.85 │         * GPT-4o-mini (Pareto: best mid-range value)
       │       
  0.80 │     * GPT-4.1-nano (Pareto: cheapest)
       │   
  0.75 │   * Model X (DOMINATED: nano is cheaper AND better)
       │
       └──────────────────────────→ Cost
```

In this example, Model X is dominated by GPT-4.1-nano (nano is cheaper and has higher quality). Model X is never the right choice. The three models on the frontier — GPT-4o, GPT-4o-mini, and GPT-4.1-nano — each serve a different budget:

| If Your Budget Is... | Choose | Why |
|---------------------|--------|-----|
| Tight (minimize cost) | GPT-4.1-nano | Cheapest model on the frontier |
| Moderate | GPT-4o-mini | Better quality than nano, still affordable |
| Unlimited (maximize quality) | GPT-4o | Highest quality, regardless of cost |

## What We Build

Using model comparison data, we identify which models are Pareto-optimal and recommend the right model for different quality thresholds and budgets.

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

"""Pareto frontier analysis from model comparison data.

Uses simulated results (replace with Demo 02 output for real data).
"""

# Simulated model results (quality, cost per query)
# Replace these with actual outputs from Demo 02
MODEL_RESULTS = {
    "gpt-4o":  {"quality": 0.95, "cost": 0.00500},
    "gpt-4o-mini":   {"quality": 0.85, "cost": 0.00030},
    "gpt-4.1-nano":       {"quality": 0.80, "cost": 0.00020},
}


def find_pareto_frontier(results: dict) -> list[str]:
    """Find models on the Pareto frontier (not dominated by any other)."""
    frontier = []
    models = list(results.items())

    for name, metrics in models:
        dominated = False
        for other_name, other_metrics in models:
            if other_name == name:
                continue
            # Other model is strictly better in both dimensions
            if other_metrics["quality"] >= metrics["quality"] and other_metrics["cost"] <= metrics["cost"]:
                if other_metrics["quality"] > metrics["quality"] or other_metrics["cost"] < metrics["cost"]:
                    dominated = True
                    break
        if not dominated:
            frontier.append(name)

    return frontier


frontier = find_pareto_frontier(MODEL_RESULTS)

print("=" * 60)
print("PARETO FRONTIER ANALYSIS")
print("=" * 60)

print(f"\n  {'Model':<18} {'Quality':<10} {'Cost/Query':<12} {'Status'}")
print(f"  {'-'*18} {'-'*10} {'-'*12} {'-'*20}")
for name, r in sorted(MODEL_RESULTS.items(), key=lambda x: x[1]["cost"]):
    status = "⭐ Pareto optimal" if name in frontier else "   Dominated"
    print(f"  {name:<18} {r['quality']:<10.2f} ${r['cost']:<11.5f} {status}")

print(f"\n📊 Pareto optimal models: {frontier}")

In [ ]:
# Recommend model based on quality threshold
THRESHOLDS = [0.70, 0.80, 0.90]

print("\n" + "=" * 60)
print("MODEL RECOMMENDATIONS BY QUALITY THRESHOLD")
print("=" * 60)

for threshold in THRESHOLDS:
    # Find cheapest model that meets threshold
    candidates = {n: r for n, r in MODEL_RESULTS.items() if r["quality"] >= threshold}
    if candidates:
        best = min(candidates.items(), key=lambda x: x[1]["cost"])
        daily_1k = best[1]["cost"] * 1000
        print(f"\n  Threshold >= {threshold}:")
        print(f"    ➡️  {best[0]} (quality={best[1]['quality']:.2f}, ${best[1]['cost']:.5f}/query)")
        print(f"    📊 At 1,000 queries/day: ${daily_1k:.2f}/day, ${daily_1k*30:.2f}/month")
    else:
        print(f"\n  Threshold >= {threshold}: No model meets this threshold")

print(f"""
  ┌────────────────────────────────────────────────────────────┐
  │ Decision Framework                                         │
  ├────────────────────────────────────────────────────────────┤
  │ Quality >= 0.90 needed?  → Use Sonnet ($8/day at 1K qps)  │
  │ Quality >= 0.80 enough?  → Use Haiku  ($1.5/day at 1K qps)│
  │ Budget is primary?       → Use Nova   ($1.2/day at 1K qps)│
  └────────────────────────────────────────────────────────────┘

  💡 Run Demo 02 with YOUR queries to get real quality scores.
     These recommendations change based on your specific tasks.
""")